In [0]:
def build_gold(name, select_sql):
    spark.sql(f"CREATE OR REPLACE TABLE zomato_ai.gold.{name} AS {select_sql}")
    print("OK", name, spark.table(f"zomato_ai.gold.{name}").count(), "rows")

In [0]:
build_gold("dim_users", """
SELECT user_id, name, email_hash, age, gender, marital_status,
       occupation, monthly_income, educational_qualifications, family_size
FROM zomato_ai.silver.users
""")

OK dim_users 100000 rows


In [0]:
build_gold("dim_restaurants", """
SELECT restaurant_id, name, city, cuisine, rating, rating_count, cost, address
FROM zomato_ai.silver.restaurants
""")

OK dim_restaurants 148544 rows


In [0]:
build_gold("dim_food", """
SELECT food_id, item, veg_or_non_veg
FROM zomato_ai.silver.food
""")

OK dim_food 371563 rows


In [0]:
build_gold("fact_orders", """
SELECT
  order_id, user_id, restaurant_id, restaurant_city, cuisine,
  order_timestamp, order_date,
  year(order_date) AS order_year,
  month(order_date) AS order_month,
  dayofweek(order_date) AS order_day_of_week,
  hour(order_timestamp) AS order_hour,
  items_count, sales_qty, subtotal, discount, delivery_fee, gst, sales_amount,
  currency, payment_method, order_status,
  (order_status = 'delivered') AS is_delivered,
  customer_rating, delivery_time_min
FROM zomato_ai.silver.orders
""")

OK fact_orders 10000000 rows


In [0]:
build_gold("fact_reviews", """
SELECT review_id, order_id, user_id, restaurant_id, rating, comment, review_date
FROM zomato_ai.silver.reviews
""")

OK fact_reviews 300000 rows


In [0]:
build_gold("agg_daily_sales", """
SELECT
  order_date,
  count(*) AS total_orders,
  sum(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
  sum(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
  sum(CASE WHEN order_status = 'refunded'  THEN 1 ELSE 0 END) AS refunded_orders,
  sum(CASE WHEN order_status = 'delivered' THEN sales_amount END) AS revenue,
  avg(delivery_time_min) AS avg_delivery_time_min,
  avg(customer_rating) AS avg_customer_rating
FROM zomato_ai.gold.fact_orders
GROUP BY order_date
""")

OK agg_daily_sales 912 rows


In [0]:
build_gold("agg_restaurant_performance", """
SELECT
  f.restaurant_id,
  r.name AS restaurant_name,
  r.city,
  count(*) AS total_orders,
  sum(CASE WHEN f.order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
  round(sum(CASE WHEN f.order_status = 'cancelled' THEN 1 ELSE 0 END) / count(*), 4) AS cancel_rate,
  sum(CASE WHEN f.order_status = 'delivered' THEN f.sales_amount END) AS revenue,
  avg(f.sales_amount) AS avg_order_value,
  avg(f.delivery_time_min) AS avg_delivery_time_min,
  avg(f.customer_rating) AS avg_customer_rating
FROM zomato_ai.gold.fact_orders f
LEFT JOIN zomato_ai.gold.dim_restaurants r ON f.restaurant_id = r.restaurant_id
GROUP BY f.restaurant_id, r.name, r.city
""")

OK agg_restaurant_performance 148541 rows


In [0]:
build_gold("agg_city_order_status", """
SELECT
  restaurant_city,
  order_status,
  count(*) AS orders,
  round(count(*) / sum(count(*)) OVER (PARTITION BY restaurant_city), 4) AS share_of_city_orders
FROM zomato_ai.gold.fact_orders
GROUP BY restaurant_city, order_status
""")

OK agg_city_order_status 2456 rows


In [0]:
# Counts should match staging for the tables copied across
pairs = [("dim_users","users"), ("dim_restaurants","restaurants"), ("dim_food","food"),
         ("fact_orders","orders"), ("fact_reviews","reviews")]
for g, s in pairs:
    gc = spark.table(f"zomato_ai.gold.{g}").count()
    sc = spark.table(f"zomato_ai.silver.{s}").count()
    print(f"{g}: {gc} vs silver {s}: {sc} -> {'OK' if gc == sc else 'MISMATCH'}")
print('----')
# Summary totals should add back up to the fact table
print("daily sales total orders:",
      spark.sql("SELECT sum(total_orders) FROM zomato_ai.gold.agg_daily_sales").collect()[0][0])
print("restaurant perf total orders:",
      spark.sql("SELECT sum(total_orders) FROM zomato_ai.gold.agg_restaurant_performance").collect()[0][0])
print("fact_orders rows:", spark.table("zomato_ai.gold.fact_orders").count())
print('----')
# Every fact row should find its dimension
spark.sql("""SELECT count(*) AS orders_without_user FROM zomato_ai.gold.fact_orders f
             LEFT ANTI JOIN zomato_ai.gold.dim_users u ON f.user_id = u.user_id""").show()
spark.sql("""SELECT count(*) AS orders_without_restaurant FROM zomato_ai.gold.fact_orders f
             LEFT ANTI JOIN zomato_ai.gold.dim_restaurants r ON f.restaurant_id = r.restaurant_id""").show()

dim_users: 100000 vs silver users: 100000 -> OK
dim_restaurants: 148544 vs silver restaurants: 148544 -> OK
dim_food: 371563 vs silver food: 371563 -> OK
fact_orders: 10000000 vs silver orders: 10000000 -> OK
fact_reviews: 300000 vs silver reviews: 300000 -> OK
----
daily sales total orders: 10000000
restaurant perf total orders: 10000000
fact_orders rows: 10000000
----
+-------------------+
|orders_without_user|
+-------------------+
|                  0|
+-------------------+

+-------------------------+
|orders_without_restaurant|
+-------------------------+
|                        0|
+-------------------------+

